# Project Vyapaar | Notebook 01: Data Cleaning 🔎

**Team:** ____________________

**Lead for this part:** Member 1 (Data Analyst)

Can we trust the data? Here we find the data problems, fix them and write down every decision.

### How to run this notebook
**On Google Colab:**
1. Upload this notebook to Colab (File → Upload notebook).
2. Click **Runtime → Run all**.
3. When the "Choose Files" button appears, select the 4 data files: `orders.csv`, `products.csv`, `reviews.csv`, `marketing.csv`.

**On Jupyter (laptop):** keep the 4 csv files in the same folder as this notebook (or in a `data` folder next to it) and run all cells.

## Step 0: Load the data

In [1]:
import pandas as pd
import plotly.express as px
import os

# show numbers with 2 decimals and commas
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)

In [2]:
# This small function finds a csv file whether it is in this folder or in a "data" folder
def find_file(name):
    for folder in ["", "data/", "../data/"]:
        if os.path.exists(folder + name):
            return folder + name
    return None

# If the files are not there we are probably on Google Colab, so ask to upload them
if find_file("orders.csv") is None:
    from google.colab import files
    print("Please upload orders.csv, products.csv, reviews.csv and marketing.csv")
    files.upload()

In [3]:
# encoding="utf-8-sig" removes a hidden character at the start of orders.csv and reviews.csv
orders = pd.read_csv(find_file("orders.csv"), encoding="utf-8-sig")
products = pd.read_csv(find_file("products.csv"))
marketing = pd.read_csv(find_file("marketing.csv"))

# keep_default_na=False stops pandas from turning the word "None" into a blank.
# In reviews, "None" means "no complaint", so we want to keep it.
reviews = pd.read_csv(find_file("reviews.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""])

print("orders:", orders.shape)
print("products:", products.shape)
print("reviews:", reviews.shape)
print("marketing:", marketing.shape)

orders: (4001, 14)
products: (121, 7)
reviews: (4001, 5)
marketing: (18, 9)


---
# Stage 1: Data Detective 🔎
Before making any recommendation, we first check if we can trust the data.

### 1.1 First look

In [4]:
orders.head()

,order_id,order_date,customer_id,product_id,quantity,unit_price_inr,discount_pct,revenue_inr,cost_inr,profit_inr,sales_channel,payment_method,campaign_id,region_at_order
0,O00001,2025-07-23,C0538,P035,1,"1,229.10",0.10,"1,106.19","1,007.86",98.33,Store,Debit Card,NaN,South
1,O00002,2025-11-22,C0015,P011,2,"6,025.06",0.00,"12,050.12","6,266.06","5,784.06",Store,COD,NaN,South
2,O00003,2025-04-16,C0158,P039,3,"1,643.71",0.20,"3,944.90","3,353.16",591.74,Marketplace,Debit Card,M01,East
3,O00004,2025-12-10,C0456,P101,1,"3,096.04",0.15,"2,631.63","1,609.94","1,021.69",Marketplace,COD,NaN,West
4,O00005,2025-03-20,C0003,P039,2,"1,643.71",0.00,"3,287.42","2,235.44","1,051.98",Website,UPI,NaN,North


In [5]:
products.head()

,product_id,product_name,category,subcategory,unit_cost_inr,list_price_inr,supplier
0,P001,Audio Item 001,Electronics,Audio,323.17,394.11,Aruna
1,P002,Footwear Item 002,Fashion,Footwear,324.66,559.76,Futura
2,P003,Fitness Item 003,Sports,Fitness,"3,265.04","5,266.19",Futura
3,P004,Sports Accessories Item 004,Sports,Sports Accessories,525.25,847.18,Futura
4,P005,Skin Care Item 005,Beauty,Skin Care,347.35,667.98,Futura


In [6]:
reviews.head()

,review_id,order_id,rating,complaint_category,escalated
0,R00001,O00001,3.00,Late delivery,0
1,R00002,O00002,4.00,None,0
2,R00003,O00003,2.00,Late delivery,1
3,R00004,O00004,4.00,None,0
4,R00005,O00005,4.00,None,0


In [7]:
marketing.head()

,campaign_id,campaign_name,channel,start_date,end_date,spend_inr,leads,conversions,cost_per_conversion_inr
0,M01,Social Media Growth 01,Social Media,2025-01-01,2025-02-15,336420,1694,138,"2,437.83"
1,M02,Search Ads Growth 02,Search Ads,2025-01-21,2025-03-07,179161,3238,242,740.33
2,M03,Email Growth 03,Email,2025-02-10,2025-03-27,209553,2660,364,575.70
3,M04,Influencer Growth 04,Influencer,2025-03-02,2025-04-16,81295,4839,272,298.88
4,M05,Affiliate Growth 05,Affiliate,2025-03-22,2025-05-06,92961,4142,320,290.50


In [8]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4001 entries, 0 to 4000
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         4001 non-null   object 
 1   order_date       4001 non-null   object 
 2   customer_id      4000 non-null   object 
 3   product_id       4001 non-null   object 
 4   quantity         4001 non-null   int64  
 5   unit_price_inr   4001 non-null   float64
 6   discount_pct     4000 non-null   float64
 7   revenue_inr      4001 non-null   float64
 8   cost_inr         4001 non-null   float64
 9   profit_inr       4001 non-null   float64
 10  sales_channel    4001 non-null   object 
 11  payment_method   4001 non-null   object 
 12  campaign_id      2529 non-null   object 
 13  region_at_order  4001 non-null   object 
dtypes: float64(5), int64(1), object(8)
memory usage: 437.7+ KB


`order_date` is stored as text (object), not as a date. We will convert it later.

### 1.2 Missing values

In [9]:
print("ORDERS")
print(orders.isnull().sum())
print()
print("REVIEWS")
print(reviews.isnull().sum())

ORDERS
order_id              0
order_date            0
customer_id           1
product_id            0
quantity              0
unit_price_inr        0
discount_pct          1
revenue_inr           0
cost_inr              0
profit_inr            0
sales_channel         0
payment_method        0
campaign_id        1472
region_at_order       0
dtype: int64

REVIEWS
review_id             0
order_id              0
rating                1
complaint_category    0
escalated             0
dtype: int64


In [10]:
# which orders have a missing customer or discount?
orders[orders["customer_id"].isnull() | orders["discount_pct"].isnull()]

,order_id,order_date,customer_id,product_id,quantity,unit_price_inr,discount_pct,revenue_inr,cost_inr,profit_inr,sales_channel,payment_method,campaign_id,region_at_order
299,O00300,2025-01-05,NaN,P058,1,"3,762.37",0.05,"3,574.25","2,332.67","1,241.58",Website,COD,M16,East
777,O00778,2025-03-06,C0179,P074,1,"3,800.24",NaN,"2,660.17","2,584.16",76.01,App,Debit Card,M11,North


In [11]:
reviews[reviews["rating"].isnull()]

,review_id,order_id,rating,complaint_category,escalated
1200,R01201,O01201,NaN,Late delivery,1


**What we decided:**
- **campaign_id** is blank for 1,472 orders. That is too many to be a mistake. It just means the order did not come from a campaign, so we label it **"No campaign"**.
- **O00300** has no customer. The sale is still real, so we keep it and call the customer **"Unknown"**.
- **O00778** has no discount. We can work it out from the revenue (see 1.6).
- **R01201** has no rating. We keep the row (it still shows a late delivery and an escalation), and we do **not** make up a rating.

### 1.3 Duplicates

In [12]:
print("Duplicate rows in orders:", orders.duplicated().sum())
print("Duplicate rows in products:", products.duplicated().sum())
print("Duplicate rows in reviews:", reviews.duplicated().sum())
print("Duplicate rows in marketing:", marketing.duplicated().sum())

Duplicate rows in orders: 1
Duplicate rows in products: 1
Duplicate rows in reviews: 1
Duplicate rows in marketing: 0


In [13]:
orders[orders.duplicated(keep=False)]

,order_id,order_date,customer_id,product_id,quantity,unit_price_inr,discount_pct,revenue_inr,cost_inr,profit_inr,sales_channel,payment_method,campaign_id,region_at_order
500,O00501,2025-08-06,C0202,P069,1,"6,706.13",0.00,"6,706.13","3,487.19","3,218.94",Website,Credit Card,M14,South
4000,O00501,2025-08-06,C0202,P069,1,"6,706.13",0.00,"6,706.13","3,487.19","3,218.94",Website,Credit Card,M14,South


In [14]:
products[products.duplicated(keep=False)]

,product_id,product_name,category,subcategory,unit_cost_inr,list_price_inr,supplier
39,P040,Desk Accessories Item 040,Office Supplies,Desk Accessories,"3,270.76","5,031.94",DigiKart
120,P040,Desk Accessories Item 040,Office Supplies,Desk Accessories,"3,270.76","5,031.94",DigiKart


The duplicates are exact copies (same ID, same values), so they are double entries and we remove them.

⚠️ The product duplicate (**P040**) is important. If we don't remove it, every P040 order will be counted **twice** when we join the tables.

### 1.4 Spelling and category problems

In [15]:
products["category"].value_counts()

category
Beauty             25
Electronics        24
Sports             20
Home & Kitchen     20
Fashion            16
Office Supplies    15
Electronic          1
Name: count, dtype: int64

In [16]:
orders["sales_channel"].value_counts()

sales_channel
App            1346
Website        1259
Store           878
Marketplace     517
web site          1
Name: count, dtype: int64

In [17]:
products[(products["category"] == "Electronic") | (products["subcategory"] == "skin care")]

,product_id,product_name,category,subcategory,unit_cost_inr,list_price_inr,supplier
10,P011,Hair Care Item 011,Electronic,Hair Care,"3,133.03","6,025.06",Crescent
73,P074,Small Appliances Item 074,Home & Kitchen,skin care,"2,584.16","3,800.24",BharatWorks


- **P011** is called "Hair Care Item 011", but its category says "Electronic".
- **P074** is called "Small Appliances Item 074", but its subcategory says "skin care".
- "web site" is just another spelling of "Website".

To be sure about the two products, we checked the **mark-up** (list price ÷ cost). Each category seems to have its own fixed mark-up:

In [18]:
products["markup"] = products["list_price_inr"] / products["unit_cost_inr"]
products.groupby("category")["markup"].mean()

category
Beauty            1.92
Electronic        1.92
Electronics       1.22
Fashion           1.72
Home & Kitchen    1.47
Office Supplies   1.54
Sports            1.61
Name: markup, dtype: float64

In [19]:
products[products["product_id"].isin(["P011", "P074"])][["product_id", "product_name", "category", "subcategory", "markup"]]

,product_id,product_name,category,subcategory,markup
10,P011,Hair Care Item 011,Electronic,Hair Care,1.92
73,P074,Small Appliances Item 074,Home & Kitchen,skin care,1.47


P011's mark-up (1.92) matches **Beauty**, and P074's (1.47) matches **Home & Kitchen**. So the names were right and the labels were wrong:
- P011 → category = **Beauty**
- P074 → subcategory = **Small Appliances**

### 1.5 Do the money columns add up?
We re-calculated revenue ourselves: **revenue = quantity × price × (1 − discount)**

In [20]:
check = orders.copy()
check["my_revenue"] = check["quantity"] * check["unit_price_inr"] * (1 - check["discount_pct"])
check["difference"] = (check["my_revenue"] - check["revenue_inr"]).abs()

print("Orders where revenue does not match:", (check["difference"] > 1).sum())
print("Orders where profit is not revenue - cost:", ((check["revenue_inr"] - check["cost_inr"] - check["profit_inr"]).abs() > 1).sum())

Orders where revenue does not match: 0
Orders where profit is not revenue - cost: 0


Everything matches 👍 so we can trust the revenue, cost and profit numbers.

### 1.6 Working out the missing discount
Since revenue = qty × price × (1 − discount), the missing discount = **1 − revenue ÷ (qty × price)**

In [21]:
row = orders[orders["discount_pct"].isnull()]
print(1 - row["revenue_inr"] / (row["quantity"] * row["unit_price_inr"]))
print()
print("Discount levels used by the company:", sorted(orders["discount_pct"].dropna().unique().tolist()))

777   0.30
dtype: float64

Discount levels used by the company: [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]


The answer is **0.30**, which is one of the company's normal discount levels (0, 5, 10, 15, 20, 30, 40%). So this is the real value, not a guess.

### 1.7 Strange values

In [22]:
print("Dates from", orders["order_date"].min(), "to", orders["order_date"].max())
print("Quantity from", orders["quantity"].min(), "to", orders["quantity"].max())
print("Ratings:", sorted(reviews["rating"].dropna().unique().tolist()))
print("Orders with NEGATIVE profit:", (orders["profit_inr"] < 0).sum())

Dates from 2025-01-01 to 2025-12-31
Quantity from 1 to 5
Ratings: [1.0, 2.0, 3.0, 4.0, 5.0]
Orders with NEGATIVE profit: 313


**313 orders have negative profit.** We first thought this was an error, but 1.5 showed the maths is correct. These are real sales where the discount was bigger than the margin, so it is a **business finding, not an error**. We keep them.

### 1.8 Do the tables connect?

In [23]:
print("Orders with a product that doesn't exist:", (~orders["product_id"].isin(products["product_id"])).sum())
print("Orders without a review:", (~orders["order_id"].isin(reviews["order_id"])).sum())

# we don't have customers.csv, so can we use region_at_order as the customer's region?
regions_per_customer = orders.groupby("customer_id")["region_at_order"].nunique()
print("Customers who ordered from more than 1 region:", (regions_per_customer > 1).sum())

Orders with a product that doesn't exist: 0
Orders without a review: 0


Customers who ordered from more than 1 region: 0


Every customer always orders from the **same region**, so we can use `region_at_order` as the customer's region (we did not get `customers.csv`).

### 1.9 Apply the cleaning

In [24]:
# 1. remove exact duplicate rows
orders = orders.drop_duplicates()
products = products.drop_duplicates()
reviews = reviews.drop_duplicates()

# 2. fix the 2 wrongly labelled products
products.loc[products["product_id"] == "P011", "category"] = "Beauty"
products.loc[products["product_id"] == "P074", "subcategory"] = "Small Appliances"
products = products.drop(columns="markup")

# 3. fix spelling of sales channel
orders["sales_channel"] = orders["sales_channel"].replace("web site", "Website")

# 4. fill the 1 missing discount using: discount = 1 - revenue / (quantity x price)
orders["discount_pct"] = orders["discount_pct"].fillna(
    (1 - orders["revenue_inr"] / (orders["quantity"] * orders["unit_price_inr"])).round(2))

# 5. label blanks instead of deleting them
orders["customer_id"] = orders["customer_id"].fillna("Unknown")
orders["campaign_id"] = orders["campaign_id"].fillna("No campaign")

# 6. text -> dates
orders["order_date"] = pd.to_datetime(orders["order_date"])
marketing["start_date"] = pd.to_datetime(marketing["start_date"])
marketing["end_date"] = pd.to_datetime(marketing["end_date"])

print("orders:", orders.shape, "| products:", products.shape, "| reviews:", reviews.shape)

orders: (4000, 14) | products: (120, 7) | reviews: (4000, 5)


### 1.10 Our cleaning log (Problem → Action → Reason)

| # | Problem found | Action taken | Reason |
|---|---|---|---|
| 1 | Hidden character at the start of orders.csv and reviews.csv | Read with `encoding="utf-8-sig"` | Otherwise the `order_id` column name is wrong and joins fail |
| 2 | pandas turns "None" (no complaint) into a blank | Read reviews with `keep_default_na=False` | "None" is real information: 2,640 orders had no complaint |
| 3 | Order O00501 appears twice | Removed 1 copy | Double entry, would add ₹6,706 of fake revenue |
| 4 | Review R00334 appears twice | Removed 1 copy | One review per order |
| 5 | Product P040 appears twice | Removed 1 copy before joining | Would double count all P040 orders |
| 6 | P011 "Hair Care Item" labelled "Electronic" | Changed to Beauty | Name + mark-up (1.92) match Beauty |
| 7 | P074 "Small Appliances Item" labelled "skin care" | Changed to Small Appliances | Name + mark-up (1.47) match Home & Kitchen |
| 8 | Sales channel "web site" | Changed to "Website" | Same channel, different spelling |
| 9 | O00778 missing discount | Calculated it = 0.30 | Worked out from revenue, so not a guess |
| 10 | O00300 missing customer | Kept, customer = "Unknown" | The sale is real |
| 11 | R01201 missing rating | Kept, rating left blank | We don't invent ratings |
| 12 | 1,472 orders with no campaign | Labelled "No campaign" | Blank means no campaign, not an error |
| 13 | 313 orders with negative profit | Kept | Maths is correct, caused by big discounts |
| 14 | Dates stored as text | Converted to dates | Needed for monthly / campaign analysis |
| 15 | customers.csv and delivery.csv not given to us | Used region_at_order and "Late delivery" complaints instead | Each customer always orders from one region |

### 1.11 One clean table

In [25]:
# join orders + products + reviews using the ID columns
df = orders.merge(products, on="product_id").merge(reviews, on="order_id")
df["margin_%"] = df["profit_inr"] / df["revenue_inr"] * 100

print("Clean table:", df.shape)
df.to_csv("vyapaar_clean.csv", index=False)   # on Colab you can download it from the folder icon
df.head()

Clean table: (4000, 25)


,order_id,order_date,customer_id,product_id,quantity,unit_price_inr,discount_pct,revenue_inr,cost_inr,profit_inr,sales_channel,payment_method,campaign_id,region_at_order,product_name,category,subcategory,unit_cost_inr,list_price_inr,supplier,review_id,rating,complaint_category,escalated,margin_%
0,O00001,2025-07-23,C0538,P035,1,"1,229.10",0.10,"1,106.19","1,007.86",98.33,Store,Debit Card,No campaign,South,Audio Item 035,Electronics,Audio,"1,007.86","1,229.10",Futura,R00001,3.00,Late delivery,0,8.89
1,O00002,2025-11-22,C0015,P011,2,"6,025.06",0.00,"12,050.12","6,266.06","5,784.06",Store,COD,No campaign,South,Hair Care Item 011,Beauty,Hair Care,"3,133.03","6,025.06",Crescent,R00002,4.00,None,0,48.00
2,O00003,2025-04-16,C0158,P039,3,"1,643.71",0.20,"3,944.90","3,353.16",591.74,Marketplace,Debit Card,M01,East,Small Appliances Item 039,Home & Kitchen,Small Appliances,"1,117.72","1,643.71",Aruna,R00003,2.00,Late delivery,1,15.00
3,O00004,2025-12-10,C0456,P101,1,"3,096.04",0.15,"2,631.63","1,609.94","1,021.69",Marketplace,COD,No campaign,West,Hair Care Item 101,Beauty,Hair Care,"1,609.94","3,096.04",DigiKart,R00004,4.00,None,0,38.82
4,O00005,2025-03-20,C0003,P039,2,"1,643.71",0.00,"3,287.42","2,235.44","1,051.98",Website,UPI,No campaign,North,Small Appliances Item 039,Home & Kitchen,Small Appliances,"1,117.72","1,643.71",Aruna,R00005,4.00,None,0,32.00


✅ **Stage 1 result:** after a few small fixes, the sales and money data can be trusted.